In [1]:
import pandas as pd
import numpy as np
import torch

import matplotlib.pyplot as plt
import plotly.graph_objects as go

import urllib3 # to download data directly from web

# Vostok Temp

[Overview of Vostok data sets](https://www.ncei.noaa.gov/access/paleo-search/study/2453)  
Called "Deuterium Data and Temperature Reconstruction  
Data: https://www.ncei.noaa.gov/pub/data/paleo/icecore/antarctica/vostok/deutnat-noaa.txt 

Variables:
- depth_m	depth,,,meter,,ice cores;climate reconstructions,,,N,
- ice_ageBP	ice age,,,calendar year before present,,ice cores;climate reconstructions,,,N,GT4 chronology
- deltaD	delta 2H,bulk ice,,per mil SMOW,,ice cores,interpolated,isotope ratio mass spectrometry,N,
- deltaTS	surface temperature,delta 2H,,degree Celsius,annual,ice cores;climate reconstructions,anomalized,,N,linear regression; anomaly from recent

In [2]:
# define link to data
url = "https://www.ncei.noaa.gov/pub/data/paleo/icecore/antarctica/vostok/deutnat-noaa.txt"

# use urllib3 package to download .txt data directly from web
# Creating a PoolManager instance for sending requests.
http = urllib3.PoolManager()

# Sending a GET request and getting back response as HTTPResponse object.
resp = http.request("GET", url)

# create new (empty) text file ('w' stands for write mode)
file = open('../data/vostok/temp/downloaded_deutnat-noaa.txt', 'w')
# populate text file with our data
file.write(resp.data.decode('utf-8-sig'))
# close file (write mode)
file.close()

# open file in read mode
file = open("../data/vostok/temp/downloaded_deutnat-noaa.txt", "r")


for index, line in enumerate(file):
    if not "#" in line:
        if line.split() == ['depth_m', 'ice_ageBP', 'deltaD', 'deltaTS']:
            # Initialise pd dataframe
            data = pd.DataFrame(columns = ('depth_m', 'ice_ageBP', 'deltaD', 'deltaTS'))
        else:
            data.loc[index] = [float(i) for i in line.split()]

# reset the index
data = data.reset_index(drop = True)
data = data[["ice_ageBP", "deltaTS"]]

### Regular interpolation

In [3]:
x_reg_400kyr = np.arange(start = 0, stop = 400000 + 1, step = 1000)
y_reg_400kyr = np.interp(x = x_reg_400kyr, xp = data["ice_ageBP"], fp = data["deltaTS"])

temp_timeseries_400kyr_vostok = torch.flip(torch.tensor(y_reg_400kyr), dims = [0])

### Non-uniqueness

In [4]:
def affirm_uniqueness(ts):
    # Check for duplicates and add noise 
    dupes = ts.shape[0] - torch.unique(ts.to(torch.float32)).shape[0]
    print("There are ", dupes, " duplicates in the timeseries.")
    if dupes > 0:
        noise_level = 0.001
        while dupes > 0:
            ts = ts + torch.randn(ts.shape[0]) * noise_level
            # recalculate dupes
            dupes = ts.shape[0] - torch.unique(ts.to(torch.float32)).shape[0]
            print("Now we have ", dupes, " dupes.")
    return ts

In [5]:
temp_timeseries_400kyr_vostok = affirm_uniqueness(temp_timeseries_400kyr_vostok)

There are  1  duplicates in the timeseries.
Now we have  0  dupes.


### Export

In [6]:
torch.save(temp_timeseries_400kyr_vostok, '../data/vostok/temp/TEMP_vostok_400kyr_timeseries.pt')

# EPICA Temp

- Data overview: https://www.ncei.noaa.gov/access/paleo-search/study/6080
- Data file: https://www.ncei.noaa.gov/pub/data/paleo/icecore/antarctica/epica_domec/edc3deuttemp2007.txt
- format of txt file is slightly different so required different code to process

In [7]:
# define link to data
url = "https://www.ncei.noaa.gov/pub/data/paleo/icecore/antarctica/epica_domec/edc3deuttemp2007.txt"

# use urllib3 package to download .txt data directly from web
# Creating a PoolManager instance for sending requests.
http = urllib3.PoolManager()

# Sending a GET request and getting back response as HTTPResponse object.
resp = http.request("GET", url)

# create new (empty) text file ('w' stands for write mode)
file = open('../data/epica/temp/downloaded_edc3deuttemp2007.txt', 'w')
# populate text file with our data
file.write(resp.data.decode('utf-8-sig'))
# close file (write mode)
file.close()

# open file in read mode
file = open("../data/epica/temp/downloaded_edc3deuttemp2007.txt", "r")

data = pd.DataFrame(columns = ('Bag', 'ztop', 'Age', 'Deuterium', 'Temperature'))

for index, line in enumerate(file):
    # Needs to be manual as file does not have #
    # skip first partial rows
    # last row is empty so stop there
    if (index > 103) & (index < 5892) & (index != 261) & (index != 310) & (index != 627): 
        data.loc[index - 103] = [float(i) for i in line.split()]

# select the two columns we need
# change integers in pandas dataframe to floats
data = data[["Age", "Temperature"]].astype('float64')

# reset the index
data = data.reset_index(drop = True)

### Regular interpolation

In [8]:
# target x at which we wan't to interpolate
x_reg = np.arange(start = 0, stop = 800000 + 1, step = 1000)

# np interpolate
y_reg = np.interp(x = x_reg, xp = data["Age"], fp = data["Temperature"])

# flip order 
TEMP_EPICA_timeseries = torch.flip(torch.tensor(y_reg), dims = [0])
TEMP_EPICA_timeseries

tensor([ -8.8983,  -8.7190,  -8.6213,  -8.6449,  -8.3792,  -7.6399,  -6.6322,
         -5.8783,  -5.3500,  -4.3422,  -3.4341,  -2.5828,  -1.8780,  -0.5853,
         -1.0417,  -1.1234,  -1.7073,  -1.9671,  -1.9598,  -2.3512,  -2.1573,
         -2.1625,  -2.4397,  -2.6938,  -2.7499,  -3.2650,  -3.7998,  -4.2659,
         -4.3488,  -3.8831,  -3.8307,  -4.6136,  -5.6004,  -6.3292,  -5.9625,
         -5.0912,  -4.7350,  -5.2339,  -6.3034,  -7.3278,  -7.1620,  -6.2366,
         -5.5243,  -5.2463,  -6.3495,  -7.4781,  -7.8825,  -8.2713,  -8.4191,
         -8.5822,  -8.3958,  -8.4979,  -8.6804,  -8.6209,  -8.5600,  -8.4810,
         -8.8216,  -8.7659,  -8.4395,  -8.2422,  -7.5558,  -5.8933,  -5.0042,
         -4.4858,  -3.8099,  -4.0990,  -3.9375,  -3.9446,  -4.7965,  -5.2534,
         -5.3379,  -5.7631,  -6.0340,  -6.0040,  -5.3482,  -5.1776,  -6.1790,
         -7.7375,  -8.5897,  -8.1336,  -7.1298,  -6.8340,  -6.3495,  -5.7071,
         -5.2458,  -4.5076,  -4.0038,  -4.1397,  -4.1713,  -4.01

### Non-uniqueness

In [9]:
TEMP_EPICA_timeseries = affirm_uniqueness(TEMP_EPICA_timeseries)

There are  0  duplicates in the timeseries.


In [10]:
torch.save(TEMP_EPICA_timeseries, '../data/epica/temp/TEMP_epica_800kyr_timeseries.pt')